# Vietnamese Malicious Comment Detection with PhoBERT
Binary classification: 0 = Clean, 1 = Malicious

## 1. Install & Import Libraries

In [ ]:
!pip install transformers underthesea -q

In [ ]:
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re

from underthesea import word_tokenize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

from transformers import (
    get_linear_schedule_with_warmup,
    AutoTokenizer,
    AutoModel,
    logging,
)

import warnings
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

In [ ]:
import torch

# Use GPU when available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Memory: {props.total_memory / 1024**3:.1f} GB")
else:
    print("GPU not available, using CPU")


In [ ]:
def seed_everything(seed_value):
    """Set seeds for reproducibility"""
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    try:
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed_value)
            torch.cuda.manual_seed_all(seed_value)
    except:
        pass

seed_everything(42)


In [ ]:
EPOCHS = 6
N_SPLITS = 5
BATCH_SIZE = 16
MAX_LEN = 128
LEARNING_RATE = 2e-5
MODEL_NAME = "vinai/phobert-base"
NUM_WORKERS = 0  # Kaggle does not support num_workers > 0 reliably

# Kaggle paths
import os
KAGGLE_INPUT = "/kaggle/input"
KAGGLE_OUTPUT = "/kaggle/working"

# Define data paths for Kaggle dataset structure
KAGGLE_DATASET_PATH = "/kaggle/input/datasets/viethoang0805/vietnamesebullydataset"

# Auto-detect: Kaggle or local
if os.path.exists(KAGGLE_DATASET_PATH):
    # Using Kaggle dataset
    DATA_CSV = os.path.join(KAGGLE_DATASET_PATH, "data.csv")
    STOPWORDS_PATH = os.path.join(KAGGLE_DATASET_PATH, "vietnamese_stopwords.txt")
    save_dir = KAGGLE_OUTPUT
elif os.path.exists(KAGGLE_INPUT):
    # Fallback: Try to find data in other Kaggle input directories
    DATA_DIR = None
    for d in os.listdir(KAGGLE_INPUT):
        candidate = os.path.join(KAGGLE_INPUT, d)
        if os.path.isfile(os.path.join(candidate, "data.csv")) or \
           os.path.isfile(os.path.join(candidate, "data", "data.csv")):
            DATA_DIR = candidate
            break
    if DATA_DIR is None:
        DATA_DIR = os.path.join(KAGGLE_INPUT, os.listdir(KAGGLE_INPUT)[0]) if os.listdir(KAGGLE_INPUT) else "."
    
    if os.path.isfile(os.path.join(DATA_DIR, "data.csv")):
        DATA_CSV = os.path.join(DATA_DIR, "data.csv")
        STOPWORDS_PATH = os.path.join(DATA_DIR, "vietnamese_stopwords.txt")
    else:
        DATA_CSV = os.path.join(DATA_DIR, "data", "data.csv")
        STOPWORDS_PATH = os.path.join(DATA_DIR, "data", "vietnamese_stopwords.txt")
    save_dir = KAGGLE_OUTPUT
else:
    # Local development
    DATA_CSV = "data/data.csv"
    STOPWORDS_PATH = "data/vietnamese_stopwords.txt"
    save_dir = "."

print(f"Data CSV      : {DATA_CSV}")
print(f"Stopwords     : {STOPWORDS_PATH}")
print(f"Output dir    : {save_dir}")

## 2. Load Dataset & Stopwords

In [ ]:
data = pd.read_csv(DATA_CSV)
data = data.iloc[:, 1:]
print(data.info())
print(f"\nLabel distribution:\n{data['label'].value_counts()}")
data.head(10)

In [ ]:
sns.countplot(x='label', data=data)
plt.title('Label Distribution')
plt.xlabel('Label (0=Clean, 1=Malicious)')
plt.ylabel('Count')
plt.show()

In [ ]:
with open(STOPWORDS_PATH, 'r', encoding='utf-8') as f:
    stopwords = set(line.strip() for line in f.readlines())

print(f"Loaded {len(stopwords)} stopwords")
print(f"Sample: {list(stopwords)[:10]}")

## 3. Text Preprocessing

In [ ]:
def preprocess_text(text, remove_stopwords=True):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^\w\s\u00C0-\u024F\u1E00-\u1EFF]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = word_tokenize(text, format="text")
    if remove_stopwords:
        words = text.split()
        words = [w for w in words if w not in stopwords]
        text = ' '.join(words)
    return text

sample = data.iloc[3]['content']
print(f"Original : {sample}")
print(f"Processed: {preprocess_text(sample)}")

In [ ]:
data['content_clean'] = data['content'].apply(preprocess_text)
data[['content', 'content_clean', 'label']].head(10)

## 4. Train/Test Split

In [ ]:
train_df = data.iloc[:int(len(data) * 0.8)].reset_index(drop=True)
test_df = data.iloc[int(len(data) * 0.8):].reset_index(drop=True)

print(f"Train size: {len(train_df)}")
print(f"Test size : {len(test_df)}")
print(f"\nTrain label distribution:\n{train_df['label'].value_counts()}")
print(f"\nTest label distribution:\n{test_df['label'].value_counts()}")

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=86)
for fold, (_, val_) in enumerate(skf.split(X=train_df, y=train_df.label)):
    train_df.loc[val_, "kfold"] = fold

train_df['kfold'] = train_df['kfold'].astype(int)
train_df.head()

## 5. Tokenizer & Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
print(f"Vocab size: {tokenizer.vocab_size}")

In [ ]:
all_texts = data['content_clean'].tolist()
encoded_texts = [tokenizer.encode(text, add_special_tokens=True) for text in all_texts]
token_lens = [len(t) for t in encoded_texts]

print(f"Max token length : {max(token_lens)}")
print(f"Mean token length: {np.mean(token_lens):.1f}")
print(f"95th percentile  : {np.percentile(token_lens, 95):.0f}")

sns.histplot(token_lens, bins=50)
plt.xlabel('Token Count')
plt.title('Distribution of Token Lengths')
plt.axvline(x=MAX_LEN, color='r', linestyle='--', label=f'MAX_LEN={MAX_LEN}')
plt.legend()
plt.show()

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.max_len = max_len
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        text = row['content_clean']
        label = row['label']

        encoding = self.tokenizer(
            text,
            truncation=True,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_attention_mask=True,
            return_token_type_ids=False,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_masks': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(label, dtype=torch.long),
        }


In [ ]:
def prepare_loaders(df, fold):
    df_train = df[df.kfold != fold].reset_index(drop=True)
    df_valid = df[df.kfold == fold].reset_index(drop=True)

    train_dataset = SentimentDataset(df_train, tokenizer, max_len=MAX_LEN)
    valid_dataset = SentimentDataset(df_valid, tokenizer, max_len=MAX_LEN)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    return train_loader, valid_loader

## 6. Model Definition

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, n_classes=2):
        super(SentimentClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.drop = nn.Dropout(p=0.3)
        self.fc = nn.Linear(self.bert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        pooled_output = outputs.pooler_output
        x = self.drop(pooled_output)
        x = self.fc(x)
        return x


## 7. Training & Evaluation Functions

In [ ]:
def train_epoch(model, criterion, optimizer, scheduler, train_loader):
    model.train()
    losses = []
    correct = 0
    total = 0

    for data in train_loader:
        input_ids = data['input_ids'].to(device)
        attention_mask = data['attention_masks'].to(device)
        targets = data['targets'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs, targets)

        _, pred = torch.max(outputs, dim=1)
        correct += torch.sum(pred == targets).item()
        total += targets.size(0)
        losses.append(loss.item())

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

    acc = correct / total
    avg_loss = np.mean(losses)
    return acc, avg_loss


def eval_epoch(model, criterion, data_loader):
    model.eval()
    losses = []
    correct = 0
    total = 0

    with torch.no_grad():
        for data in data_loader:
            input_ids = data['input_ids'].to(device)
            attention_mask = data['attention_masks'].to(device)
            targets = data['targets'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, targets)

            _, pred = torch.max(outputs, dim=1)
            correct += torch.sum(pred == targets).item()
            total += targets.size(0)
            losses.append(loss.item())

    acc = correct / total
    avg_loss = np.mean(losses)
    return acc, avg_loss

## 8. K-Fold Training

In [ ]:
fold_results = []

for fold in range(N_SPLITS):
    print(f'\n{"="*60}')
    print(f'FOLD {fold + 1}/{N_SPLITS}')
    print(f'{"="*60}')

    train_loader, valid_loader = prepare_loaders(train_df, fold=fold)

    model = SentimentClassifier(n_classes=2).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps,
    )

    best_val_acc = 0
    for epoch in range(EPOCHS):
        train_acc, train_loss = train_epoch(model, criterion, optimizer, scheduler, train_loader)
        val_acc, val_loss = eval_epoch(model, criterion, valid_loader)

        print(f'Epoch {epoch+1}/{EPOCHS} | '
              f'Train Acc: {train_acc:.4f} Loss: {train_loss:.4f} | '
              f'Val Acc: {val_acc:.4f} Loss: {val_loss:.4f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            os.makedirs(save_dir, exist_ok=True)
            torch.save(model.state_dict(), os.path.join(save_dir, f'phobert_fold{fold+1}.pth'))

    fold_results.append(best_val_acc)
    print(f'Best Val Accuracy for Fold {fold+1}: {best_val_acc:.4f}')

print(f'\n{"="*60}')
print(f'Average Val Accuracy: {np.mean(fold_results):.4f} (+/- {np.std(fold_results):.4f})')


## 9. Test Evaluation (Ensemble)

In [ ]:
def test_ensemble(test_df, tokenizer, n_folds=N_SPLITS):
    models = []
    for fold in range(n_folds):
        try:
            model_path = os.path.join(save_dir, f'phobert_fold{fold+1}.pth')
            if os.path.exists(model_path):
                m = SentimentClassifier(n_classes=2).to(device)
                m.load_state_dict(torch.load(model_path, map_location=device))
                m.eval()
                models.append(m)
        except Exception as e:
            print(f"Warning: Could not load fold {fold+1}: {e}")

    if not models:
        print("Error: No models loaded!")
        return None, None, None

    test_dataset = SentimentDataset(test_df, tokenizer, max_len=MAX_LEN)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    all_preds = []
    all_targets = []
    all_texts = []

    with torch.no_grad():
        for data in test_loader:
            input_ids = data['input_ids'].to(device)
            attention_mask = data['attention_masks'].to(device)
            targets = data['targets']

            fold_outputs = []
            for m in models:
                outputs = m(input_ids=input_ids, attention_mask=attention_mask)
                fold_outputs.append(outputs)

            avg_outputs = torch.stack(fold_outputs).mean(0)
            _, preds = torch.max(avg_outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.numpy())
            all_texts.extend(data['text'])

    return np.array(all_targets), np.array(all_preds), all_texts


In [ ]:
print("Evaluating on test set...")
real_values, predicts, texts = test_ensemble(test_df, tokenizer)

if real_values is not None:
    print("\n" + "="*60)
    print(classification_report(real_values, predicts, target_names=['Clean (0)', 'Malicious (1)']))
    print("="*60)
else:
    print("Could not evaluate - models not available yet")


## 10. Confusion Matrix

In [ ]:
if real_values is not None:
    cm = confusion_matrix(real_values, predicts)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Clean (0)', 'Malicious (1)'],
                yticklabels=['Clean (0)', 'Malicious (1)'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix - PhoBERT Ensemble')
    plt.show()
else:
    print("Cannot create confusion matrix - run test evaluation first")


## 11. Error Analysis

In [ ]:
if real_values is not None:
    wrong_indices = np.where(predicts != real_values)[0]
    print(f"Total wrong predictions: {len(wrong_indices)} / {len(real_values)}")
    print(f"Error rate: {len(wrong_indices)/len(real_values)*100:.2f}%\n")

    for i in wrong_indices[:15]:
        print('-' * 60)
        print(f"Text: {test_df.iloc[i]['content']}")
        print(f"Predicted: {predicts[i]} | Actual: {real_values[i]}")
else:
    print("Cannot analyze errors - run test evaluation first")


## 12. Inference

In [ ]:
def predict(text, tokenizer, model, device):
    """Predict sentiment for a given text"""
    label_map = {0: 'Clean', 1: 'Malicious'}
    processed = preprocess_text(text)

    encoding = tokenizer(
        processed,
        max_length=MAX_LEN,
        truncation=True,
        add_special_tokens=True,
        padding='max_length',
        return_attention_mask=True,
        return_token_type_ids=False,
        return_tensors='pt',
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    model.eval()
    with torch.no_grad():
        output = model(input_ids, attention_mask)
        probs = torch.softmax(output, dim=1)
        confidence, pred = torch.max(probs, dim=1)

    label = label_map[pred.item()]
    conf = confidence.item()
    
    print(f"Text       : {text}")
    print(f"Prediction : {label} (confidence: {conf:.4f})")
    return label, conf


In [ ]:
if len(fold_results) > 0:
    best_fold = np.argmax(fold_results) + 1
    model_path = os.path.join(save_dir, f'phobert_fold{best_fold}.pth')
    
    if os.path.exists(model_path):
        print(f"Loading best model from fold {best_fold}")
        best_model = SentimentClassifier(n_classes=2).to(device)
        best_model.load_state_dict(torch.load(model_path, map_location=device))
        best_model.eval()

        print("\n" + "="*60)
        print("Testing on sample sentences:")
        print("="*60)
        
        test_sentences = [
            "Ban lam tot lam, co gang nhe!",
            "Thang ngu, do vo dung",
            "Hom nay troi dep qua",
            "Do con cho, may bien di",
            "Cam on ban rat nhieu",
        ]

        for sentence in test_sentences:
            predict(sentence, tokenizer, best_model, device)
            print()
    else:
        print(f"Model file not found: {model_path}")
else:
    print("No fold results available - train the model first")


## 13. Save Model & Tokenizer

In [ ]:
if len(fold_results) > 0:
    best_fold = np.argmax(fold_results) + 1
    model_path = os.path.join(save_dir, f'phobert_fold{best_fold}.pth')
    
    if os.path.exists(model_path):
        output_dir = os.path.join(save_dir, "phobert_malicious_detection")
        os.makedirs(output_dir, exist_ok=True)

        # Load and save best model
        best_model = SentimentClassifier(n_classes=2).to(device)
        best_model.load_state_dict(torch.load(model_path, map_location=device))
        torch.save(best_model.state_dict(), os.path.join(output_dir, 'phobert_best.pth'))
        
        # Save tokenizer
        tokenizer.save_pretrained(output_dir)

        print(f"✓ Model and tokenizer saved to: {output_dir}")
        print(f"\nFiles saved:")
        for f in sorted(os.listdir(output_dir)):
            file_path = os.path.join(output_dir, f)
            file_size = os.path.getsize(file_path) / (1024 * 1024)
            print(f"  - {f} ({file_size:.1f} MB)")
    else:
        print(f"Model file not found: {model_path}")
else:
    print("No trained models available - complete training first")
